# Phase 1: Detection Agent Evaluation

This notebook evaluates the performance of the Detection Agent using Isolation Forest and heuristic rules.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from sklearn.metrics import PrecisionRecallDisplay, confusion_matrix

# Add src to path
sys.path.append(os.path.abspath('..'))

from src.pipeline.data_ingestion import load_ibm_pipeline, generate_synthetic_data
from src.agents.detection_agent import DetectionAgent

sns.set_theme(style="whitegrid")

## 1. Load Data & Instantiate Agent

In [ ]:
DATA_PATH = "../data/raw/HI-Small_Trans.csv"
if os.path.exists(DATA_PATH):
    df = load_ibm_pipeline(DATA_PATH)
else:
    print("Raw data not found. Using synthetic data for evaluation.")
    synthetic_raw = generate_synthetic_data(10000)
    synthetic_raw.to_csv("temp_eval.csv", index=False)
    df = load_ibm_pipeline("temp_eval.csv")
    os.remove("temp_eval.csv")

agent = DetectionAgent(contamination=0.03)

## 2. Train Model

In [ ]:
agent.train(df, force_retrain=True)
print("Model trained successfully.")

## 3. Detect Anomalies

In [ ]:
df_results = agent.detect(df)
flagged = df_results[df_results['is_flagged']]
print(f"Flagged {len(flagged)} transactions out of {len(df)}.")
print(f"Flag Rate: {len(flagged)/len(df):.4f}")
print("\nFlag Reason Breakdown:")
print(flagged['flag_reason'].value_counts())

## 4. Evaluate Metrics

In [ ]:
metrics = agent.evaluate(df_results)

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(df_results['is_laundering'], df_results['is_flagged'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Clean', 'Suspicious'], 
            yticklabels=['Clean', 'Suspicious'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 6. Precision-Recall Curve

In [ ]:
pd_display = PrecisionRecallDisplay.from_predictions(
    df_results['is_laundering'], df_results['anomaly_score'] * -1, name="Detection Agent"
)
plt.title("Precision-Recall Curve")
plt.show()

## 7. Anomaly Score Distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df_results, x='anomaly_score', hue='is_laundering', element="step", stat="density", common_norm=False, alpha=0.5)
plt.title("Anomaly Score Distribution (Clean vs Laundering)")
plt.axvline(x=agent.flag_threshold, color='red', linestyle='--', label='Threshold')
plt.legend()
plt.show()

## 8. Flag Reason Breakdown

In [ ]:
plt.figure(figsize=(10, 6))
reason_counts = flagged['flag_reason'].value_counts()
sns.barplot(x=reason_counts.values, y=reason_counts.index, hue=reason_counts.index, palette="magma")
plt.title("Flag Reason Distribution")
plt.xlabel("Count")
plt.show()

## 9. Save Results

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
df_results[df_results['is_flagged']].to_csv("../data/processed/flagged_transactions.csv", index=False)
df_results.to_csv("../data/processed/clean_transactions.csv", index=False)
print("Results saved to data/processed/")